# deckard Model Inversion (MNIST)

This notebook reproduces the ART MIFace MNIST model inversion flow using deckard configs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from deckard.attack.torch_utils import is_tensor, tensor_to_numpy
from deckard.data.pytorch import PytorchDataConfig
from deckard.model.pytorch import PytorchModelConfig

from deckard.attack import AttackConfig

In [ ]:
def as_numpy(x):
    if is_tensor(x):
        return tensor_to_numpy(x)
    return np.asarray(x)

data_conf = PytorchDataConfig(
    dataset_name='torch_mnist',
    train_size=1024,
    test_size=256,
    random_state=42,
    classifier=True,
    stratify=True,
    data_dir='raw_data/torch_mnist',
)
data_conf()

model_conf = PytorchModelConfig(
    model_type='torch_example.py:ResNet18',
    model_params={'num_channels': 1, 'num_classes': 10},
    classifier=True,
    criterion='CrossEntropyLoss',
    optimizer={'name': 'SGD', 'lr': 0.01, 'momentum': 0.9},
    fit_params={'nb_epochs': 1, 'batch_size': 256, 'verbose': False},
    clip_values=[0, 255],
)
train_scores = model_conf(data_conf)
train_scores

In [ ]:
targets = np.arange(10, dtype=int)

def run_model_inversion(initialization):
    attack = AttackConfig(
        attack_type='art.attacks.inference.model_inversion.mi_face.MIFace',
        attack_size=10,
        attack_params={
            'max_iter': 300,
            'threshold': 1.0,
            'initialization': initialization,
            'targets': targets.tolist(),
            'split': 'test',
        },
    )
    scores = attack(data_conf, model_conf)
    return attack.predictions, scores

inits = ['zeros', 'average', 'ones', 'random']
results = {}
for init in inits:
    recon, scores = run_model_inversion(init)
    results[init] = {'recon': as_numpy(recon), 'scores': scores}

{k: v['scores'] for k, v in results.items()}

In [ ]:
def plot_recon_grid(recon, title):
    recon = np.asarray(recon)
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    axes = axes.ravel()
    for i in range(min(10, len(recon))):
        img = recon[i]
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = np.moveaxis(img, 0, -1)
        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f'class {i}')
        axes[i].axis('off')
    fig.suptitle(title)
    fig.tight_layout()

for init in inits:
    plot_recon_grid(results[init]['recon'], f'MIFace reconstruction ({init})')